In [0]:
dbutils.widgets.text("catalog","fk_1361")
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql import functions as F

In [0]:
df = spark.read.table(f"{catalog}.bronze.stores_bronze")

In [0]:
df.display()

In [0]:
def standardize(df):
    return df.withColumns({
        "store_id" : F.trim(F.upper(F.col("store_id"))),
        "store_name" : F.trim(F.initcap(F.col("store_name"))),
        "city" : F.trim(F.initcap(F.col("city"))),
        "state" : F.trim(F.initcap(F.col("state"))),
        "region" : F.trim(F.initcap(F.col("region"))),
        "opening_date" : F.to_date(F.col("opening_date"), "yyyy-MM-dd"),
    })

In [0]:
df_cleaned = standardize(df)
df_cleaned = df_cleaned.dropna(subset=["store_id","opening_date"]).dropDuplicates()
df_cleaned.write.mode("overwrite").saveAsTable(f"{catalog}.silver.stores_silver")

In [0]:
df_qurantine = standardize(df)
df_qurantine = df_qurantine.filter(F.col("store_id").isNull() | F.col("opening_date").isNull())
df_qurantine.write.mode("overwrite").saveAsTable(f"{catalog}.silver.stores_qurantine")

In [0]:
%sql
select count(*) from fk_1361.bronze.stores_bronze

In [0]:
%sql
select count(*) from fk_1361.silver.stores_silver